# Session 12: ウェイポイント飛行
# Session 12: Waypoint Mission

## 目標 / Objective

複数点を経由する自律飛行を実装し、軌跡精度を定量評価する。

Implement autonomous multi-waypoint flight and quantitatively evaluate trajectory accuracy.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| ウェイポイント計画 | 経由点の設定と順序 |
| 到達判定 | 閾値による切り替え |
| 精度評価 | RMSE, 最大偏差 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stampfly_edu import connect_or_simulate, load_sample_data, plot_trajectory

print("Ready! / 準備完了！")

## 1. ウェイポイントミッション計画 / Mission Planning

```
WP3(0,1.5) ─────── WP2(1,1.5)
     │                   │
     │                   │
WP4(0,0.5) ─────── WP1(1,0.5)
                         ↑
                    HOME(0.5,0)
```

In [ ]:
# Define waypoints (x, y in meters)
# ウェイポイント定義 (x, y メートル)
waypoints = [
    (0.5, 0.0),   # Home
    (1.0, 0.5),   # WP1
    (1.0, 1.5),   # WP2
    (0.0, 1.5),   # WP3
    (0.0, 0.5),   # WP4
    (0.5, 0.0),   # Return home
]

# Visualize mission plan
fig, ax = plt.subplots(figsize=(8, 8))
wx = [p[0] for p in waypoints]
wy = [p[1] for p in waypoints]

ax.plot(wx, wy, 'r--o', markersize=10, linewidth=2, label='Planned path / 計画経路')
for i, (x, y) in enumerate(waypoints):
    label = 'Home' if i == 0 else f'WP{i}'
    ax.annotate(label, (x, y), textcoords='offset points',
                xytext=(10, 10), fontsize=10)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Mission Plan / ミッション計画')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 2. ミッション実行 / Mission Execution

In [ ]:
# Execute waypoint mission with simulator
# シミュレータでウェイポイントミッションを実行
drone = connect_or_simulate()
drone.takeoff()

# Convert waypoints to SDK commands
# ウェイポイントを SDK コマンドに変換
for i in range(1, len(waypoints)):
    dx = waypoints[i][0] - waypoints[i-1][0]
    dy = waypoints[i][1] - waypoints[i-1][1]
    
    # Convert to cm
    dx_cm = int(abs(dx * 100))
    dy_cm = int(abs(dy * 100))
    
    if dx_cm >= 20:
        if dx > 0:
            drone.move_forward(dx_cm)
        else:
            drone.move_back(dx_cm)
    
    if dy_cm >= 20:
        if dy > 0:
            drone.move_left(dy_cm)
        else:
            drone.move_right(dy_cm)
    
    label = f'WP{i}' if i < len(waypoints)-1 else 'Home'
    print(f"Reached {label} ({waypoints[i][0]:.1f}, {waypoints[i][1]:.1f})")

drone.land()
print("Mission complete! / ミッション完了！")

## 3. ログ解析 / Log Analysis

In [ ]:
# Use sample data for analysis
# サンプルデータで分析
log = load_sample_data("square_path")
ax = plot_trajectory(log, title="Waypoint Mission Result / ウェイポイントミッション結果")
plt.show()

## 4. 考察課題 / Discussion Questions

1. **到達判定閾値**: 閾値を大きくするとミッション時間はどう変わるか？精度は？
2. **経路最適化**: 最短経路と安全性のトレードオフをどう考えるか？
3. **風の影響**: 一定方向の風があるとき、復路の精度はどう変わるか？

---

1. **Arrival Threshold**: How does a larger threshold affect mission time? Accuracy?
2. **Path Optimization**: How to balance shortest path vs safety?
3. **Wind Effect**: How does constant wind affect return path accuracy?

In [ ]:
drone.end()